# Free GPU codec-compute probe

**Goal:** Verify real NVIDIA NVDEC access on a free Colab GPU and run the repository's software arithmetic proof. Hardware H.264 decoding is not itself a proof of general-purpose arithmetic on NVDEC.

1. Choose **Runtime → Change runtime type → T4 GPU** (if offered).
2. Run all cells.
3. Inspect the printed diagnostics and download `codec_probe_results.zip` from the Colab Files sidebar.

If a GPU or codec library is unavailable, the notebook reports the actual failure rather than claiming success. No user data or proprietary source files are used.


In [ ]:
import json, pathlib, shutil, subprocess, sys, urllib.request, zipfile

def probe(argv):
    completed = subprocess.run(argv, text=True, capture_output=True)
    return {"command": argv, "returncode": completed.returncode,
            "stdout": completed.stdout[-5000:], "stderr": completed.stderr[-5000:]}

print(json.dumps(probe(['nvidia-smi','-L']) if shutil.which('nvidia-smi') else {"error": "nvidia-smi absent; no GPU runtime"}, indent=2))
print(json.dumps(probe(['ffmpeg','-hide_banner','-decoders']) if shutil.which('ffmpeg') else {"error": "ffmpeg missing"}, indent=2)[-3500:])


## Fetch the pinned GitHub experiment

The source is pinned to a specific Git commit for reproducibility; it does not fetch the moving default branch.


In [ ]:
SOURCE_COMMIT = 'd43afc08924bfb32fbba588abb76955fb9ef8ff0'
URL = f'https://raw.githubusercontent.com/DeerSpotter/Test/{SOURCE_COMMIT}/experiments/codec-compute/run.py'
source = urllib.request.urlopen(URL, timeout=30).read()
path = pathlib.Path('/content/codec_compute_run.py')
path.write_bytes(source)
print(f'Retrieved {len(source)} bytes from {URL}')


## Run the CPU proof and strict hardware decoder test

The CPU test must reconstruct all 4,096 modular additions exactly. The NVIDIA test explicitly invokes `h264_cuvid` and must fail if NVDEC cannot be reached. Timings are end-to-end and include FFmpeg process startup; do not interpret them as codec-kernel-only benchmarks.


In [ ]:
results = {}
for mode in ('cpu', 'nvdec'):
    cmd = [sys.executable, str(path), '--mode', mode, '--out', f'/content/{mode}_result.json']
    record = probe(cmd)
    results[mode] = record
    print('\n###', mode.upper(), 'exit', record['returncode'])
    print(record['stdout'][-3500:])
    print(record['stderr'][-5000:])

bundle = pathlib.Path('/content/codec_probe_results.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr('execution_diagnostics.json', json.dumps(results, indent=2))
    for mode in results:
        p = pathlib.Path(f'/content/{mode}_result.json')
        if p.exists(): archive.write(p, arcname=p.name)
print('\nSaved diagnostic bundle:', bundle)
if results['cpu']['returncode'] != 0:
    raise RuntimeError('CPU proof failed; exact diagnostics printed and saved before failure.')
if results['nvdec']['returncode'] != 0:
    raise RuntimeError('NVDEC was not accessible or the strict hardware test failed; diagnostics printed and saved before failure.')
print('Both tests passed. Reminder: this establishes hardware video decoding, NOT arithmetic via NVDEC.')
